# 13.6: Uniform quantization

- **Per-tensor:** one scale for the whole matrix. Cheapest, but most sensitive to outliers.
- **Per-channel:** one scale per output row. Standard for INT8 weights.
- **Per-group:** one scale per group of, say, 64 or 128 consecutive weights. Standard for 4-bit weights. The scales add a little storage (for groups of 128 with 16-bit scales, 16/128 = 0.125 extra bits per weight).

The following code quantizes a random matrix and measures the error in a matrix-vector product:


In [ ]:
import numpy as np

def quantize_per_channel(W, bits=8):
    """Symmetric per-output-channel quantization of a weight matrix W [out, in]."""
    qmax = 2 ** (bits - 1) - 1                      # 127 for INT8, 7 for INT4
    scale = np.abs(W).max(axis=1, keepdims=True) / qmax
    Q = np.clip(np.round(W / scale), -qmax, qmax).astype(np.int8)
    return Q, scale

def dequantize(Q, scale):
    return Q.astype(np.float32) * scale

rng = np.random.default_rng(0)
W = rng.standard_normal((4096, 4096)).astype(np.float32) * 0.02
x = rng.standard_normal(4096).astype(np.float32)

for bits in [8, 4]:
    Q, s = quantize_per_channel(W, bits)
    y, y_hat = W @ x, dequantize(Q, s) @ x
    rel_err = np.linalg.norm(y - y_hat) / np.linalg.norm(y)
    print(f"INT{bits}: relative output error {rel_err:.4f}")

def quantize_per_group(W, bits=4, group=128):
    """Symmetric quantization with one scale per group of `group` input weights."""
    out_dim, in_dim = W.shape
    Wg = W.reshape(out_dim, in_dim // group, group)
    qmax = 2 ** (bits - 1) - 1
    scale = np.abs(Wg).max(axis=2, keepdims=True) / qmax
    Q = np.clip(np.round(Wg / scale), -qmax, qmax).astype(np.int8)
    return Q, scale

Q, s = quantize_per_group(W, bits=4, group=128)
y_hat = (Q.astype(np.float32) * s).reshape(W.shape) @ x
print(f"INT4, groups of 128: relative output error {np.linalg.norm(W @ x - y_hat) / np.linalg.norm(W @ x):.4f}")
